# Python Advanced - Day 4 Homework

**Runbook alignment (Day 4, Hours 13-16):**
- Hour 13: HTTP client work: requests + JSON contracts
- Hour 14: Security basics - environment variables, safe secrets habits, and hashing concepts
- Hour 15: Capstone planning workshop - scope, milestones, and delivery path
- Hour 16: Checkpoint 2: Persistence-ready core + JSON save/load

This homework connects the tracker app to HTTP-style JSON workflows, secure configuration habits, and capstone planning decisions.



### Autograder Integration Requirements

This assignment is graded automatically by the **Global Autograder**.

- All solution code must live in this notebook and run without errors.
- The autograder converts the submission notebook to `advanced_day4.py` and grades it against `criteria.json`.
- Required outputs must exactly match the labeled canonical strings in `criteria.json`.
- Keep output deterministic: no randomness, live timestamps, network calls, or hidden external state.
- Do not rename this notebook or move the config files (`criteria.json`, `setup.json`, `feedback.json`).
- Keep file access inside the assignment directory only.
- Run all cells before submitting.



## Part 1: Hour 13 - HTTP client work: requests + JSON contracts

**Goal:** Build a client call that reads JSON safely and prints labeled contract checks.

**Best practice:** Check status codes and JSON keys before trusting remote data.

**Avoid this pitfall:** Assuming every response has the shape you want leads to fragile clients.


In [ ]:
from dataclasses import dataclass

@dataclass
class FakeResponse:
    url: str
    status_code: int
    payload: dict[str, object]
    timeout_used: bool

    def json(self) -> dict[str, object]:
        return self.payload

def fetch_tasks() -> FakeResponse:
    return FakeResponse(
        url="https://api.example.test/tasks",
        status_code=200,
        payload={"id": "task-101", "title": "Draft syllabus", "priority": "medium"},
        timeout_used=True,
    )

response = fetch_tasks()
json_keys = ", ".join(response.json().keys())

print(f"Hour 13 | request url: {response.url}")
print(f"Hour 13 | response status: {response.status_code}")
print(f"Hour 13 | json keys: {json_keys}")
print(f"Hour 13 | timeout used: {'yes' if response.timeout_used else 'no'}")
print("Hour 13 | contract rule: validate fields before use")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 2: Hour 14 - Security basics - environment variables, safe secrets habits, and hashing concepts

**Goal:** Build a secure config pattern that separates secrets from code.

**Best practice:** Read secrets from environment variables and store only hashes when appropriate.

**Avoid this pitfall:** Checking secrets into source control is never acceptable.


In [ ]:
import hashlib
import os

os.environ["TRACKER_SECRET"] = "safe-demo-secret"
secret_source = "environment variable" if os.getenv("TRACKER_SECRET") else "missing"
password_hash = hashlib.sha256("demo-password".encode("utf-8")).hexdigest()
plain_password_stored = False

print(f"Hour 14 | secret source: {secret_source}")
print("Hour 14 | hardcoded secret: rejected")
print("Hour 14 | hash algorithm: sha256")
print(f"Hour 14 | plain password stored: {'yes' if plain_password_stored else 'no'}")
print("Hour 14 | security rule: smallest safe scope first")
assert len(password_hash) == 64


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.


## Part 3: Hour 15 - Capstone planning workshop - scope, milestones, and delivery path

**Goal:** Build a concrete plan artifact and a behavior check for delivery readiness.

**Definition of Done (used by grading):**
1. Create `planning_artifacts/hour15_capstone_plan.json` with keys: `domain`, `interface`, `milestones` (>=2), `scope_guardrail`, `definition_of_done` (>=3).
2. Create `planning_artifacts/hour15_rubric.md` that includes a `Definition of Done` section.
3. Implement `plan_ready_for_build(plan: dict) -> bool` that returns `True` for a complete plan and `False` for an incomplete plan.
            


In [ ]:
import json
from pathlib import Path

capstone_plan = {
    "domain": "tracker",
    "interface": "gui",
    "milestones": [
        "domain + service layer",
        "persistence + reporting",
    ],
    "scope_guardrail": "one core record type",
    "definition_of_done": [
        "core CRUD works",
        "JSON persistence works",
        "demo path documented",
    ],
}


def plan_ready_for_build(plan: dict) -> bool:
    required_text_fields = ["domain", "interface", "scope_guardrail"]
    has_required_text = all(isinstance(plan.get(key), str) and plan.get(key).strip() for key in required_text_fields)
    milestones = plan.get("milestones", [])
    definition = plan.get("definition_of_done", [])
    return has_required_text and isinstance(milestones, list) and len(milestones) >= 2 and isinstance(definition, list) and len(definition) >= 3


artifacts_dir = Path("planning_artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
(artifacts_dir / "hour15_capstone_plan.json").write_text(json.dumps(capstone_plan, indent=2), encoding="utf-8")
(artifacts_dir / "hour15_rubric.md").write_text(
    "# Hour 15 Rubric\n\n## Definition of Done\n- Domain and interface selected\n- Milestones are sequenced\n- Scope guardrail is explicit\n",
    encoding="utf-8",
)

print(f"Hour 15 | plan ready: {plan_ready_for_build(capstone_plan)}")
            


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 4: Hour 16 - Checkpoint 2: Persistence-ready core + JSON save/load

**Goal:** Build a core app that can add records and round-trip them through JSON.

**Best practice:** Checkpoint the core model and save or load path before expanding the interface.

**Avoid this pitfall:** Delaying persistence checks until after major UI work makes bugs harder to isolate.


In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import json


@dataclass
class Task:
    title: str
    priority: str


class JsonTaskService:
    def __init__(self, storage_path: Path) -> None:
        self._storage_path = storage_path
        self._tasks: list[Task] = []

    def add(self, task: Task) -> None:
        self._tasks.append(task)

    def save(self) -> None:
        self._storage_path.parent.mkdir(parents=True, exist_ok=True)
        self._storage_path.write_text(
            json.dumps([asdict(task) for task in self._tasks], indent=2),
            encoding="utf-8",
        )

    def load(self) -> list[Task]:
        raw_items = json.loads(self._storage_path.read_text(encoding="utf-8"))
        self._tasks = [Task(**item) for item in raw_items]
        return self._tasks.copy()


base_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
storage_path = base_dir / "checkpoint_data" / "tasks.json"
checkpoint_service = JsonTaskService(storage_path)
checkpoint_service.add(Task("Ship demo", "high"))
checkpoint_service.save()
loaded_tasks = checkpoint_service.load()

print("Hour 16 | checkpoint model: Task")
print("Hour 16 | checkpoint service add: ok")
print("Hour 16 | checkpoint json save: ok")
print("Hour 16 | checkpoint json load: ok")
print("Hour 16 | checkpoint result: persistence-ready core")
assert loaded_tasks[0].title == "Ship demo"


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.


## Reflection Questions

1. Which hour felt strongest today, and why?
2. Which output label was hardest to make deterministic?
3. What would you improve before a checkpoint or demo?

### Your Answers


## Submission Checklist

- My completed notebook stays named `Advanced_Day4_homework.ipynb`.
- I copied a finished submission notebook into `submissions/` before grading.
- My output labels match `criteria.json` exactly.
- I avoided randomness, live timestamps, and hidden external state.
- I ran all cells and fixed errors before submission.


## Grading Note

Hour 15 now validates artifact files plus `plan_ready_for_build` behavior.
            
